In [ ]:
import json
from pathlib import Path

all_chunks = []

books = {
    "أصول السنة لأحمد بن حنبل.json": ("أصول السنة", "أحمد بن حنبل", "الحنبلي", "عقيدة"),
    "الاقتصاد في الاعتقاد للغزالي.json": ("الاقتصاد في الاعتقاد", "الغزالي", "الشافعي", "عقيدة"),
    "التوحيد للماتريدي.json": ("التوحيد", "الماتريدي", "الحنفي", "عقيدة"),
    "الفقه الأكبر.json": ("الفقه الأكبر", "أبو حنيفة", "الحنفي", "عقيدة"),
    "عقيدة السلف - مقدمة أبي زيد القيرواني لكتابه الرسالة.json": ("عقيدة السلف", "أبو زيد القيرواني", "المالكي", "عقيدة"),
    "سيرة ابن هشام ت السقا.json": ("السيرة النبوية", "ابن هشام", "", "سيرة"),
}

for file, (book, author, madhhab, category) in books.items():
    with open(Path("data") / file, "r", encoding="utf-8") as f:
        chunks = json.load(f)

    for chunk in chunks:
        chunk["metadata"]["book"] = book
        chunk["metadata"]["author"] = author
        chunk["metadata"]["مذهب"] = madhhab
        chunk["metadata"]["category"] = category
        all_chunks.append(chunk)




In [3]:
%pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------------------------------------- -- 1.6/1.7 MB 10.5 MB/s eta 0:00:01
   ---------------------------------------- 1.7/1.7 MB 10.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [6]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

texts = [
    "passage: " + chunk["text"]
    for chunk in all_chunks
]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True
)
#for chunk, embedding in zip(all_chunks, embeddings):
#    chunk["embedding"] = embedding.tolist()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/271 [00:00<?, ?it/s]

: 

<h1>Using Fatwa Data</h1>

In [21]:
def remove_stopwords_from_text(text, stopwords):
    if not isinstance(text, str):
        return ""

    words = text.split()
    filtered = [w for w in words if w not in stopwords]
    return " ".join(filtered)


In [22]:
import re

def normalize_arabic(text):
    """
    Normalize Arabic text by:
    - Removing diacritics
    - Removing tatweel
    - Normalizing Alef forms to ا
    - Normalizing ي / ى
    - Normalizing ة to ه
    - Normalizing ئؤ to ء
    - Removing punctuation
    - Normalizing whitespace
    """

    if not isinstance(text, str):
        return ""

    # ------------------------------------------------------
    # 1. Remove Arabic diacritics & tatweel
    # ------------------------------------------------------
    arabic_diacritics = re.compile(r"""
        ّ    |  # Shadda
        َ    |  # Fatha
        ً    |  # Tanwin Fath
        ُ    |  # Damma
        ٌ    |  # Tanwin Damm
        ِ    |  # Kasra
        ٍ    |  # Tanwin Kasr
        ْ    |  # Sukun
        ـ       # Tatweel
    """, re.VERBOSE)

    text = re.sub(arabic_diacritics, "", text)

    # ------------------------------------------------------
    # 2. Remove punctuation
    # ------------------------------------------------------
    punctuations = r"""!.,;:()'"؟،«»…"""
    text = re.sub(f"[{re.escape(punctuations)}]", " ", text)

    # ------------------------------------------------------
    # 3. Normalize whitespace
    # ------------------------------------------------------
    text = re.sub(r"\s+", " ", text).strip()

    # ------------------------------------------------------
    # 4. Normalize Arabic letters
    # ------------------------------------------------------
    text = re.sub("[أإآٱ]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = re.sub("[ئؤ]", "ء", text)

    # ------------------------------------------------------
    # 5. Final cleanup
    # ------------------------------------------------------
    text = text.replace("ـ", "")

    return text


In [23]:
import json
from pathlib import Path

def safe_text(value):
    return value if isinstance(value, str) else ""

def transform_fatwa_files(folder_path, stopwords):
    all_chunks = []

    fatwa_files = [
        "001_quran.json",
        "002_quran.json",
        "003_quran.json",
        "004_quran.json",
        "005_quran.json",
        "006_quran.json",
        "007_quran.json",
        "008_quran.json",
        "009_quran.json",
        "010_quran.json",
        "011_quran.json",
        "012_quran.json",
    ]

    folder = Path(folder_path)

    for file_name in fatwa_files:
        file_path = folder / file_name

        if not file_path.exists():
            continue

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        for item in data:
            raw_title = safe_text(item.get("title"))
            raw_question = safe_text(item.get("question"))
            raw_answer = safe_text(item.get("answer"))

            # -------- Normalize (except ayat)
            norm_question = normalize_arabic(raw_question)
            norm_answer = normalize_arabic(raw_answer)

            # -------- Remove stopwords (embedding only)
            clean_question = remove_stopwords_from_text(norm_question, stopwords)
            clean_answer = remove_stopwords_from_text(norm_answer, stopwords)

            # -------- Build embedding text
            text_parts = []

            if raw_title:
                text_parts.append(f"عنوان الفتوى:\n{raw_title}")

            if clean_question:
                text_parts.append(f"سؤال:\n{clean_question}")

            if clean_answer:
                text_parts.append(f"الجواب:\n{clean_answer}")

            if not text_parts:
                continue

            embedding_text = "\n\n".join(text_parts)

            chunk = {
                "text": embedding_text,  # CLEANED → for embedding
                "metadata": {
                    "type": "fatwa",
                    "fatwa_number": item.get("fatwa_number"),
                    "title": raw_title,
                    "question_raw": raw_question,
                    "answer_raw": raw_answer,
                    "source_file": file_name,
                    "category": "فتاوى معاصرة",
                    "quran_references": item.get("quran_references", [])
                }
            }

            all_chunks.append(chunk)

    return all_chunks


In [24]:
# -----------------------------------------------------------
# LOAD ARABIC STOP WORDS
# -----------------------------------------------------------
def load_arabic_stopwords(filepath):
    """
    Reads Arabic stop words from a file (one word per line)
    and returns them as a set.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        stopwords = {line.strip() for line in f if line.strip()}
    return stopwords
arabic_stopwords = load_arabic_stopwords("arabic_stopwords_quran.txt")

In [25]:
fatwa_chunks = transform_fatwa_files("data/Contemporary_fatawa",arabic_stopwords)

In [26]:
print(fatwa_chunks[2])  # Print the first 2 chunks for verification

{'text': 'عنوان الفتوى:\nالرقية بالقرآن الكريم والأدعية المشروعة\n\nسؤال:\nونصه لوحظ الاونه الاخيره قيام بعض الادعياء والزاعمين بالاعلان الصحف المحليه مدعين قدرتهم علي العلاج بالقران الكريم مستغلين جهل البسطاء وسهوله خداعهم واكل اموالهم بالباطل ولا يخفي عليكم ان القران الكريم دستور خالد نزل لهدايه الناس والتشريع وتنظيم عباداتهم ومعاملاتهم وقد جاءت جميع ايات القران الكريم مءكده الحقيقه؛ قوله تعالي {كتاب انزلناه اليك مبارك ليدبروا اياته} [ص 29] {وما انزلنا عليك الكتاب الا لتبين الذي اختلفوا فيه} [النحل 64] {ذلك الكتاب ريب فيه هدي للمتقين} [البقره 2] وغير الايات الكثيره الداله علي مراد الشارع الحكيم انزال الكتب السماويه وخاصه القران الكريم ورغبه حفظ كتاب الله امتهان الادعياء والزاعمين ونايا ان يستغل وسيله لكسب مشروع فانني التمس سيادتكم اصدار فتوي الخصوص؛ تبين الضوابط الشرعيه لهذا العمل لتكون مرجعا لمن اراد معرفه الحكم الشرعي\n\nالجواب:\nذهب جمهور الفقهاء الي جواز الرقيه بالقران داء يصيب الانسان لقوله تعالي {وننزل القران شفاء ورحمه للمءمنين} [الاسراء 82] ولما اخرجه البخاري عبد الرحمن بن ال

In [27]:
all_chunks.extend(fatwa_chunks)
with open(Path("data") / "all_books_and_fatawa_combined.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, ensure_ascii=False, indent=2)